# Task 2 - Incremental CPG Parser Service

The service uses Python `ast`, stable structural IDs,
statement-level CFG, lexical reaching-definitions DFG, and conservative
top-level same-file call resolution. It releases each file graph before
processing the next file.

```mermaid
flowchart LR
  F[One Python file] --> A[Python ast]
  A --> N[AST nodes and edges]
  A --> C[Statement CFG]
  C --> D[Reaching-definitions DFG]
  A --> K[Conservative CALL edges]
  N --> T[One Kafka transaction]
  C --> T
  D --> T
  K --> T
  T --> M[Manifest advances after commit]
```

## Parser strategy and its limits

Python's standard `ast` module was the pragmatic choice for this repository: it
preserves every syntax node required by the lab and introduces no separate
runtime. Joern could provide deeper interprocedural analysis, and tree-sitter
would be attractive for mixed Python versions, but either choice would add
integration work without removing the need to define stable IDs and replay
semantics ourselves.

Incrementality is enforced at the file boundary. A file is decoded, analyzed,
compared with its previous stable node and edge sets, and published in one Kafka
transaction before the SQLite manifest advances. This order is important:
updating the manifest first could make a failed Kafka transaction look complete.
Processing and then releasing one graph bounds analysis memory by the largest
file rather than by the repository.

The semantic passes are deliberately conservative. Structural AST paths define
identity, CFG edges represent statement-level control, and a bounded
reaching-definitions fixed point supplies DFG edges within each lexical scope.
When a definition or call target cannot be proved locally, the graph records an
external node instead of guessing. Alias analysis, precise exception dispatch,
and dynamic method resolution remain explicit limitations.

In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

root = Path('..').resolve()
sys.path.insert(0, str(root))
from cpg_parser.analyzer import CPGAnalyzer
from cpg_parser.discovery import discover_repo
from cpg_parser.ids import file_id

repo = root / 'source-repo'
if not repo.is_dir():
    repo = root.parent / 'source-repo'
report = discover_repo(repo)
node_counts, edge_counts = Counter(), Counter()
node_ids, edge_ids = set(), set()
warnings = 0
for relative in report.files:
    result = CPGAnalyzer(
        (repo / relative).read_text(encoding='utf-8', errors='replace'),
        file_id('huggingface/optimum', relative), relative,
    ).analyze()
    node_counts.update(result.node_counts())
    edge_counts.update(result.edge_counts())
    node_ids.update(node.id for node in result.nodes)
    edge_ids.update(edge.id for edge in result.edges)
    warnings += len(result.warnings)
summary = {
    'repository': 'huggingface/optimum',
    'files': len(report.files),
    'nodes': sum(node_counts.values()),
    'edges': sum(edge_counts.values()),
    'node_counts': dict(sorted(node_counts.items())),
    'edge_counts': dict(sorted(edge_counts.items())),
    'warnings': warnings,
    'all_node_ids_unique': len(node_ids) == sum(node_counts.values()),
    'all_edge_ids_unique': len(edge_ids) == sum(edge_counts.values()),
}
print(json.dumps(summary, indent=2))
assert {'AST', 'CFG', 'DFG', 'CALL'} <= set(edge_counts)
assert summary['all_node_ids_unique'] and summary['all_edge_ids_unique']
print('PASS: all CPG categories exist and IDs are unique')

{
  "repository": "huggingface/optimum",
  "files": 61,
  "nodes": 62550,
  "edges": 77873,
  "node_counts": {
    "AST": 58021,
    "EXTERNAL": 3047,
    "SYNTHETIC": 1482
  },
  "edge_counts": {
    "AST": 57960,
    "CALL": 2593,
    "CFG": 7248,
    "DFG": 10072
  },
  "warnings": 31,
  "all_node_ids_unique": true,
  "all_edge_ids_unique": true
}
PASS: all CPG categories exist and IDs are unique


In [2]:
import subprocess
import sys
from pathlib import Path

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q'],
    cwd=Path('..').resolve(), capture_output=True, text=True, check=True,
)
print(result.stdout.rstrip())
assert '[100%]' in result.stdout
print('PASS: parser, replay, schema, and syntax-error tests')

.........................                                                [100%]
PASS: parser, replay, schema, and syntax-error tests


## Reflection

The regression run produced deterministic IDs and all four CPG
edge categories while processing files one at a time. Two weaknesses surfaced
during fixture testing. A syntax error could leave the previous valid graph
visible, and the first DFG implementation mishandled augmented assignments and
deletions. Error transactions now remove stale topology before publishing error
metadata, while the transfer functions model the implicit read in `x += value`
and the kill caused by `del`. Attribute calls still remain external because
resolving them without type information would create more misleading edges than
useful ones.